# GA vs Random Search — Stroke Prediction

This revised notebook compares Genetic Algorithm (GA) and Random Search (RS) on the same stroke dataset.

**Experimental design:** fold-specific preprocessing through a `Pipeline`; no oversampling; repeated stratified 10-fold outer cross-validation (2 repeats); stratified 3-fold inner cross-validation; MCC as the tuning objective; a planned budget of 210 candidates per optimizer; additional evaluation metrics, runtime, best parameters, out-of-fold predictions, confidence intervals, and paired statistical tests.

> **Note:** The actual number of GA candidate evaluations is recorded from `cv_results_` and must be checked. The `population_size` and `generations` parameters do not automatically guarantee exactly 210 evaluations in every library version. Run the candidate-count audit before concluding that the two optimizers used identical search budgets.


In [ ]:
# ============================================================
# 1. ENVIRONMENT & IMPORTS
# ============================================================
# Install packages only if they are not already available. If Kaggle Internet access is unavailable,
# make sure the packages are available through Kaggle Input or offline wheels.
import importlib
import subprocess
import sys
import time
import json
import warnings
from pathlib import Path

REQUIRED = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scikit-learn": "sklearn",
    "scipy": "scipy",
    "sklearn-genetic-opt": "sklearn_genetic",
    "xgboost": "xgboost",
    "matplotlib": "matplotlib",
}

missing = []
for distribution, module in REQUIRED.items():
    try:
        importlib.import_module(module)
    except Exception:
        missing.append(distribution)

if missing:
    print("Attempting to install packages that could not be imported:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import numpy as np
import pandas as pd
import scipy
import sklearn
import xgboost
import matplotlib
from scipy.stats import randint, uniform, loguniform, wilcoxon, rankdata
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve, auc,
    confusion_matrix, balanced_accuracy_score, matthews_corrcoef
)
from sklearn_genetic import GASearchCV
from sklearn_genetic.space import Integer, Continuous, Categorical

warnings.filterwarnings("ignore", category=UserWarning)

print("Python:", sys.version)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("scipy:", scipy.__version__)
print("xgboost:", xgboost.__version__)
try:
    from importlib.metadata import version
    print("sklearn-genetic-opt:", version("sklearn-genetic-opt"))
except Exception:
    print("sklearn-genetic-opt: version metadata unavailable")


In [ ]:
# ============================================================
# 2. CONFIGURATION & DATA
# ============================================================
SEED = 42
N_OUTER_SPLITS = 10
N_REPEATS = 2
N_INNER_SPLITS = 3

# Planned search budget. RS evaluates up to 100 sampled candidates.
# GA parameters are set to target ~100 candidates; ACTUAL evaluations
# are saved from cv_results_ and must be inspected in the output CSV.
SEARCH_BUDGET = 210
GA_POPULATION_SIZE = 10
GA_GENERATIONS = 10
RS_N_ITER = SEARCH_BUDGET

# Use the raw dataset so preprocessing is fitted inside each training fold.
DATA_PATH = "/kaggle/input/datasets/aderahmad/stroke-dataset-raw/healthcare-dataset-stroke-data.csv"
OUTPUT_DIR = Path("/kaggle/working/ga_vs_rs_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
if "id" in df.columns:
    df = df.drop(columns=["id"])

TARGET = "stroke"
if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' was not found. Available columns: {df.columns.tolist()}")

X = df.drop(columns=[TARGET]).copy()
y = pd.to_numeric(df[TARGET], errors="raise").astype(int)

if set(y.unique()) != {0, 1}:
    raise ValueError(f"Target must be binary (0/1). Found: {sorted(y.unique().tolist())}")

print("Dataset shape:", df.shape)
print("Target distribution:")
print(y.value_counts().sort_index())
print("Positive prevalence:", round(float(y.mean()), 5))

# Column definitions aligned with the revised preprocessing workflow.
NUMERIC_FEATURES = ["age", "hypertension", "heart_disease", "avg_glucose_level", "bmi"]
ONEHOT_FEATURES = ["gender", "work_type", "smoking_status"]
ORDINAL_FEATURES = ["ever_married", "Residence_type"]

expected = set(NUMERIC_FEATURES + ONEHOT_FEATURES + ORDINAL_FEATURES)
missing_features = expected - set(X.columns)
if missing_features:
    raise ValueError(f"Expected features were not found: {sorted(missing_features)}")

# Do not impute/encode the full dataset here. Every transformer is fitted
# only on each inner-training split by the sklearn Pipeline.


In [ ]:
# ============================================================
# 3. FOLD-SPECIFIC PREPROCESSING & MODEL SEARCH SPACES
# ============================================================
def make_preprocessor(scale_numeric=False):
    numeric_steps = [("imputer", SimpleImputer(strategy="mean"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    numeric_pipe = Pipeline(numeric_steps)

    onehot_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    # Binary encoding: ever_married No/Yes -> 0/1;
    # Residence_type Rural/Urban -> 0/1.
    # Encoded within the fold-specific preprocessing pipeline.
    ordinal_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(
            categories=[["No", "Yes"], ["Rural", "Urban"]],
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ))
    ])

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipe, NUMERIC_FEATURES),
            ("onehot", onehot_pipe, ONEHOT_FEATURES),
            ("ordinal", ordinal_pipe, ORDINAL_FEATURES),
        ],
        remainder="drop",
        verbose_feature_names_out=False
    )

# Search-space definitions. Each model has equivalent GA and RS domains.
# class_weight / scale_pos_weight are explicitly included where applicable.
MODEL_CONFIGS = {
    "Random Forest": {
        "estimator": RandomForestClassifier(random_state=SEED, n_jobs=1),
        "scale_numeric": False,
        "ga_space": {
            "model__n_estimators": Integer(50, 300),
            "model__max_depth": Integer(3, 10),
            "model__min_samples_split": Integer(2, 10),
            "model__min_samples_leaf": Integer(5, 10),
            "model__class_weight": Categorical(["balanced", "balanced_subsample", None]),
        },
        "rs_space": {
            "model__n_estimators": randint(50, 301),
            "model__max_depth": randint(3, 11),
            "model__min_samples_split": randint(2, 11),
            "model__min_samples_leaf": randint(5, 11),
            "model__class_weight": ["balanced", "balanced_subsample", None],
        },
    },
    "XGBoost": {
        "estimator": XGBClassifier(
            random_state=SEED,
            eval_metric="logloss",
            objective="binary:logistic",
            n_jobs=1,
            tree_method="hist"
        ),
        "scale_numeric": False,
        "ga_space": {
            "model__n_estimators": Integer(50, 300),
            "model__max_depth": Integer(3, 10),
            "model__learning_rate": Continuous(0.01, 0.3),
            "model__subsample": Continuous(0.5, 1.0),
            "model__colsample_bytree": Continuous(0.5, 1.0),
            "model__scale_pos_weight": Continuous(1, 25),
        },
        "rs_space": {
            "model__n_estimators": randint(50, 301),
            "model__max_depth": randint(3, 11),
            "model__learning_rate": uniform(0.01, 0.29),
            "model__subsample": uniform(0.5, 0.5),
            "model__colsample_bytree": uniform(0.5, 0.5),
            "model__scale_pos_weight": uniform(1, 24),
        },
    },
    "Logistic Regression": {
        "estimator": LogisticRegression(max_iter=1000, random_state=SEED),
        "scale_numeric": True,
        "ga_space": {
            "model__C": Continuous(0.01, 10.0, distribution="log-uniform"),
            "model__penalty": Categorical(["l2"]),
            "model__solver": Categorical(["lbfgs", "saga"]),
            "model__class_weight": Categorical([None, "balanced"]),
        },
        "rs_space": {
            "model__C": loguniform(0.01, 10.0),
            "model__penalty": ["l2"],
            "model__solver": ["lbfgs", "saga"],
            "model__class_weight": [None, "balanced"],
        },
    },
    "SVM": {
        "estimator": SVC(probability=True, random_state=SEED),
        "scale_numeric": True,
        "ga_space": {
            "model__C": Continuous(0.01, 10.0, distribution="log-uniform"),
            "model__kernel": Categorical(["rbf"]),
            "model__gamma": Categorical(["scale", "auto"]),
            "model__class_weight": Categorical([None, "balanced"]),
        },
        "rs_space": {
            "model__C": loguniform(0.01, 10.0),
            "model__kernel": ["rbf"],
            "model__gamma": ["scale", "auto"],
            "model__class_weight": [None, "balanced"],
        },
    },
}

# Model set and hyperparameter spaces match the configurations supplied
# from the ablation notebook.
MODELS_TO_RUN = [
    "Random Forest",
    "XGBoost",
    "Logistic Regression",
    "SVM",
]
OPTIMIZERS = ["GA", "RandomSearch"]

print("Models:", MODELS_TO_RUN)
print("Optimizers:", OPTIMIZERS)
print("Planned candidate budget per search:", SEARCH_BUDGET)

In [ ]:
# ============================================================
# 4. METRICS, SEARCH FACTORY, AND CHECKPOINT HELPERS
# ============================================================
def calculate_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    precision, recall_curve, _ = precision_recall_curve(y_true, y_prob)
    # Trapezoidal area under the precision-recall curve (distinct from AP).
    pr_auc_value = auc(recall_curve, precision)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "average_precision": average_precision_score(y_true, y_prob),
        "pr_auc": pr_auc_value,
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }

def build_pipeline(model_name):
    cfg = MODEL_CONFIGS[model_name]
    return Pipeline([
        ("preprocess", make_preprocessor(scale_numeric=cfg["scale_numeric"])),
        ("model", clone(cfg["estimator"]))
    ])

def make_optimizer(optimizer_name, model_name, pipeline, inner_cv, fold_seed):
    cfg = MODEL_CONFIGS[model_name]
    common = dict(
        estimator=pipeline,
        cv=inner_cv,
        scoring="matthews_corrcoef",
        n_jobs=-1,
        refit=True,
        error_score="raise",
    )

    if optimizer_name == "GA":
        return GASearchCV(
            **common,
            param_grid=cfg["ga_space"],
            population_size=GA_POPULATION_SIZE,
            generations=GA_GENERATIONS,
            crossover_probability=0.2,
            mutation_probability=0.8,
            elitism=True,
            keep_top_k=1,
            criteria="max",
            algorithm="eaMuPlusLambda",
            verbose=False
        )

    if optimizer_name == "RandomSearch":
        return RandomizedSearchCV(
            **common,
            param_distributions=cfg["rs_space"],
            n_iter=RS_N_ITER,
            random_state=fold_seed,
            return_train_score=False
        )

    raise ValueError(f"Unknown optimizer: {optimizer_name}")

def append_csv(records, filename):
    if not records:
        return
    frame = pd.DataFrame(records)
    target_path = OUTPUT_DIR / filename
    if target_path.exists():
        frame.to_csv(target_path, mode="a", header=False, index=False)
    else:
        frame.to_csv(target_path, index=False)

def bootstrap_mean_ci(values, n_boot=5000, seed=SEED):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    samples = rng.choice(values, size=(n_boot, len(values)), replace=True).mean(axis=1)
    return float(np.quantile(samples, 0.025)), float(np.quantile(samples, 0.975))

print("Helper functions ready.")


In [ ]:
# ============================================================
# 5. EXPERIMENT: PAIRED OUTER SPLITS FOR GA AND RANDOM SEARCH
# ============================================================
# A single fixed outer split list is shared by both optimizers. This makes
# fold-level performance comparisons paired. All preprocessing is fitted
# inside each inner/outer training subset by the Pipeline.
outer_cv = RepeatedStratifiedKFold(
    n_splits=N_OUTER_SPLITS,
    n_repeats=N_REPEATS,
    random_state=SEED
)
outer_splits = list(outer_cv.split(X, y))

fold_records = []
prediction_records = []
parameter_records = []
search_audit_records = []

# Clear old outputs to prevent accidentally mixing results from different runs.
for filename in [
    "ga_rs_fold_results.csv",
    "ga_rs_predictions.csv",
    "ga_rs_best_parameters.csv",
    "ga_rs_search_audit.csv",
]:
    f = OUTPUT_DIR / filename
    if f.exists():
        f.unlink()

total_jobs = len(MODELS_TO_RUN) * len(OPTIMIZERS) * len(outer_splits)
job_counter = 0

for model_name in MODELS_TO_RUN:
    for split_id, (train_idx, test_idx) in enumerate(outer_splits, start=1):
        repeat_id = (split_id - 1) // N_OUTER_SPLITS + 1
        outer_fold = (split_id - 1) % N_OUTER_SPLITS + 1

        X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
        y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

        # Same inner fold definition/seed for both optimizers within this outer split.
        inner_cv = StratifiedKFold(
            n_splits=N_INNER_SPLITS,
            shuffle=True,
            random_state=SEED + split_id
        )

        for optimizer_name in OPTIMIZERS:
            job_counter += 1
            fold_seed = SEED + split_id

            print(
                f"[{job_counter}/{total_jobs}] {model_name} | "
                f"{optimizer_name} | repeat={repeat_id}, fold={outer_fold}"
            )

            pipeline = build_pipeline(model_name)
            optimizer = make_optimizer(
                optimizer_name, model_name, pipeline, inner_cv, fold_seed
            )

            start = time.perf_counter()
            optimizer.fit(X_train, y_train)
            tuning_and_refit_seconds = time.perf_counter() - start

            best_model = optimizer.best_estimator_
            y_pred = best_model.predict(X_test)
            y_prob = best_model.predict_proba(X_test)[:, 1]
            metrics = calculate_metrics(y_test, y_pred, y_prob)

            # Actual candidate count is obtained from the fitted search object.
            cv_results = getattr(optimizer, "cv_results_", {})
            actual_candidate_count = len(cv_results.get("params", []))
            unique_candidate_count = len({
                json.dumps(p, sort_keys=True, default=str)
                for p in cv_results.get("params", [])
            })

            row = {
                "Model": model_name,
                "Optimizer": optimizer_name,
                "Repeat": repeat_id,
                "Outer_Fold": outer_fold,
                "Split_ID": split_id,
                "Inner_CV_Folds": N_INNER_SPLITS,
                "Search_Scoring": "MCC",
                "Best_Inner_CV_MCC": float(optimizer.best_score_),
                "Tuning_And_Refit_Seconds": tuning_and_refit_seconds,
                "Actual_Candidate_Evaluations": actual_candidate_count,
                "Unique_Candidate_Configurations": unique_candidate_count,
                "Planned_Budget": SEARCH_BUDGET,
                "Target_Prevalence_Test": float(y_test.mean()),
                **metrics,
            }
            fold_records.append(row)
            append_csv([row], "ga_rs_fold_results.csv")

            params_row = {
                "Model": model_name,
                "Optimizer": optimizer_name,
                "Repeat": repeat_id,
                "Outer_Fold": outer_fold,
                "Split_ID": split_id,
                "Best_Inner_CV_MCC": float(optimizer.best_score_),
                "Best_Params_JSON": json.dumps(optimizer.best_params_, default=str, sort_keys=True),
            }
            parameter_records.append(params_row)
            append_csv([params_row], "ga_rs_best_parameters.csv")

            audit_row = {
                "Model": model_name,
                "Optimizer": optimizer_name,
                "Repeat": repeat_id,
                "Outer_Fold": outer_fold,
                "Split_ID": split_id,
                "Planned_Budget": SEARCH_BUDGET,
                "Actual_Candidate_Evaluations": actual_candidate_count,
                "Unique_Candidate_Configurations": unique_candidate_count,
                "Inner_CV_Folds": N_INNER_SPLITS,
                "Best_Inner_CV_MCC": float(optimizer.best_score_),
                "Elapsed_Seconds": tuning_and_refit_seconds,
            }
            search_audit_records.append(audit_row)
            append_csv([audit_row], "ga_rs_search_audit.csv")

            for idx, (true_label, pred_label, prob) in enumerate(zip(y_test, y_pred, y_prob)):
                prediction_row = {
                    "Model": model_name,
                    "Optimizer": optimizer_name,
                    "Repeat": repeat_id,
                    "Outer_Fold": outer_fold,
                    "Split_ID": split_id,
                    "Test_Row_Index": int(test_idx[idx]),
                    "y_true": int(true_label),
                    "y_pred": int(pred_label),
                    "y_prob": float(prob),
                }
                prediction_records.append(prediction_row)
            append_csv(
                [r for r in prediction_records[-len(test_idx):]],
                "ga_rs_predictions.csv"
            )

            print(
                f"  MCC={metrics['mcc']:.4f}; "
                f"Recall={metrics['recall_sensitivity']:.4f}; "
                f"PR-AUC={metrics['pr_auc']:.4f}; "
                f"elapsed={tuning_and_refit_seconds:.1f}s; "
                f"candidates={actual_candidate_count} "
                f"(unique={unique_candidate_count})"
            )

print("\nEksperimen selesai. Output folder:", OUTPUT_DIR)
print("Note: elapsed time includes cross-validation search and refitting the best estimator.")


In [ ]:
# ============================================================
# 6. SUMMARY: MEAN, SD, AND DESCRIPTIVE 95% BOOTSTRAP CI
# ============================================================
fold_df = pd.read_csv(OUTPUT_DIR / "ga_rs_fold_results.csv")

METRICS = [
    "accuracy", "precision", "recall_sensitivity", "specificity", "f1",
    "roc_auc", "average_precision", "pr_auc", "balanced_accuracy", "mcc",
    "Tuning_And_Refit_Seconds"
]

summary_rows = []
for (model_name, optimizer_name), group in fold_df.groupby(["Model", "Optimizer"]):
    for metric in METRICS:
        vals = group[metric].dropna().to_numpy(dtype=float)
        ci_low, ci_high = bootstrap_mean_ci(vals, seed=SEED)
        summary_rows.append({
            "Model": model_name,
            "Optimizer": optimizer_name,
            "Metric": metric,
            "N_outer_splits": len(vals),
            "Mean": float(np.mean(vals)) if len(vals) else np.nan,
            "SD": float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan,
            "CI95_bootstrap_low_descriptive": ci_low,
            "CI95_bootstrap_high_descriptive": ci_high,
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / "ga_rs_summary.csv", index=False)
display(summary_df)

# Explicitly show whether planned candidate budgets were actually matched.
audit_df = pd.read_csv(OUTPUT_DIR / "ga_rs_search_audit.csv")
budget_summary = (
    audit_df.groupby(["Model", "Optimizer"])
    .agg(
        Mean_actual_candidates=("Actual_Candidate_Evaluations", "mean"),
        Min_actual_candidates=("Actual_Candidate_Evaluations", "min"),
        Max_actual_candidates=("Actual_Candidate_Evaluations", "max"),
        Mean_unique_candidates=("Unique_Candidate_Configurations", "mean"),
        Mean_elapsed_seconds=("Elapsed_Seconds", "mean"),
    )
    .reset_index()
)
display(budget_summary)
budget_summary.to_csv(OUTPUT_DIR / "ga_rs_budget_summary.csv", index=False)

print(
    "CI interpretation: the bootstrap intervals above are descriptive. "
    "Outer-fold scores in repeated CV are dependent; therefore, these CIs "
    "are not inferential intervals that fully account for fold dependence."
)


In [ ]:
# ============================================================
# 7. PAIRED STATISTICAL COMPARISON: GA VS RANDOM SEARCH
# ============================================================
# Paired by Model + Split_ID (same outer test split). Wilcoxon is applied
# separately per model and metric. Report effect size (matched-pairs
# rank-biserial correlation) and bootstrap CI for mean paired difference.
#
# Caution: repeated-CV fold scores are dependent. Interpret these tests as
# fold-paired evidence, not as if every fold were an independent dataset.

def paired_rank_biserial(differences):
    d = np.asarray(differences, dtype=float)
    d = d[np.isfinite(d)]
    d = d[d != 0]
    if len(d) == 0:
        return 0.0
    ranks = rankdata(np.abs(d))
    w_pos = ranks[d > 0].sum()
    w_neg = ranks[d < 0].sum()
    denom = w_pos + w_neg
    return float((w_pos - w_neg) / denom) if denom else 0.0

def paired_bootstrap_ci(differences, n_boot=5000, seed=SEED):
    d = np.asarray(differences, dtype=float)
    d = d[np.isfinite(d)]
    if len(d) == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    samples = rng.choice(d, size=(n_boot, len(d)), replace=True).mean(axis=1)
    return float(np.quantile(samples, 0.025)), float(np.quantile(samples, 0.975))

paired_rows = []
for model_name in MODELS_TO_RUN:
    model_df = fold_df[fold_df["Model"] == model_name]
    ga = model_df[model_df["Optimizer"] == "GA"].set_index("Split_ID")
    rs = model_df[model_df["Optimizer"] == "RandomSearch"].set_index("Split_ID")

    common_ids = ga.index.intersection(rs.index)
    ga, rs = ga.loc[common_ids].sort_index(), rs.loc[common_ids].sort_index()

    for metric in [
        "mcc", "balanced_accuracy", "recall_sensitivity",
        "specificity", "f1", "average_precision", "pr_auc",
        "roc_auc", "Tuning_And_Refit_Seconds"
    ]:
        ga_values = ga[metric].to_numpy(dtype=float)
        rs_values = rs[metric].to_numpy(dtype=float)
        differences = ga_values - rs_values
        valid = np.isfinite(differences)
        differences = differences[valid]

        if len(differences) >= 2 and np.any(differences != 0):
            try:
                stat, p_value = wilcoxon(
                    ga_values[valid], rs_values[valid],
                    alternative="two-sided", zero_method="wilcox",
                    method="auto"
                )
            except Exception:
                stat, p_value = np.nan, np.nan
        else:
            stat, p_value = np.nan, np.nan

        ci_low, ci_high = paired_bootstrap_ci(
            differences, seed=SEED + len(paired_rows)
        )
        paired_rows.append({
            "Model": model_name,
            "Metric": metric,
            "N_paired_outer_splits": len(differences),
            "Mean_GA": float(np.nanmean(ga_values)),
            "Mean_RandomSearch": float(np.nanmean(rs_values)),
            "Mean_difference_GA_minus_RS": float(np.nanmean(differences)),
            "Difference_CI95_bootstrap_low_descriptive": ci_low,
            "Difference_CI95_bootstrap_high_descriptive": ci_high,
            "Wilcoxon_statistic": stat,
            "p_value_raw": p_value,
            "Effect_size_rank_biserial": paired_rank_biserial(differences),
        })

paired_df = pd.DataFrame(paired_rows)

# Holm correction across the tested metrics within each model.
paired_df["p_value_holm_within_model"] = np.nan
for model_name, idx in paired_df.groupby("Model").groups.items():
    pvals = paired_df.loc[idx, "p_value_raw"].to_numpy(dtype=float)
    finite = np.isfinite(pvals)
    if finite.any():
        p = pvals[finite]
        order = np.argsort(p)
        adjusted = np.empty_like(p)
        m = len(p)
        running_max = 0.0
        for rank_idx, original_idx in enumerate(order):
            candidate = min(1.0, (m - rank_idx) * p[original_idx])
            running_max = max(running_max, candidate)
            adjusted[original_idx] = running_max
        adjusted_back = np.full(len(pvals), np.nan)
        adjusted_back[np.where(finite)[0]] = adjusted
        paired_df.loc[idx, "p_value_holm_within_model"] = adjusted_back

paired_df.to_csv(OUTPUT_DIR / "ga_rs_paired_statistics.csv", index=False)
display(paired_df)

print(
    "Statistical caution: because repeated outer CV reuses observations across "
    "test folds, paired Wilcoxon p-values and fold-bootstrap CIs should be "
    "reported with this limitation. Do not describe folds as independent cohorts."
)


In [ ]:
# ============================================================
# 8. FINAL VALIDATION CHECKS & FILES
# ============================================================
print("Output files:")
for file_path in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"- {file_path.name}: {file_path.stat().st_size:,} bytes")

print("\nBudget equivalence check:")
audit = pd.read_csv(OUTPUT_DIR / "ga_rs_search_audit.csv")
display(
    audit.groupby(["Model", "Optimizer"])[
        ["Actual_Candidate_Evaluations", "Unique_Candidate_Configurations"]
    ].agg(["mean", "min", "max"])
)

print(
    "\nBefore writing the manuscript, verify for each model that GA and RS "
    "have comparable actual candidate counts. If GA does not evaluate the "
    "same number of candidates as RS, adjust GA generations/population and "
    "rerun; do not claim equal search budgets based only on configured values."
)
